In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/test_queries.csv
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/sample_submission.csv
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/documents.csv
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/dataset-metadata.json
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/train_queries.csv
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/qrels_train.csv
/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers/baseline_submission.csv


In [2]:
!pip install sentence_transformers rank_bm25 -q

import os
import glob
import torch
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder, util

print("🚀 Starting Phase 2 Pipeline: HyDE + Multi-Query Expansion + Dual Reranker...")

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"⚡ Device initialized: {device}")

# -------------------------------------------------------------------------
# 1. LOAD DATASETS
# -------------------------------------------------------------------------
input_files = glob.glob('/kaggle/input/**/documents.csv', recursive=True)
input_dir = os.path.dirname(input_files[0]) if input_files else '/kaggle/input/competitions/agricultural-extension-rag-smart-retrieval-for-farmers'

documents_df = pd.read_csv(os.path.join(input_dir, 'documents.csv'))
test_queries_df = pd.read_csv(os.path.join(input_dir, 'test_queries.csv'))

doc_text_col = 'document_text' if 'document_text' in documents_df.columns else ('text' if 'text' in documents_df.columns else documents_df.columns[1])
doc_id_col = 'document_id' if 'document_id' in documents_df.columns else documents_df.columns[0]
query_id_col_test = 'query_id' if 'query_id' in test_queries_df.columns else test_queries_df.columns[0]
test_query_col = 'query_text' if 'query_text' in test_queries_df.columns else ('query' if 'query' in test_queries_df.columns else test_queries_df.columns[1])

def build_doc_string(row):
    title = str(row['title']) if 'title' in row and pd.notna(row['title']) else ''
    cat = str(row['category']) if 'category' in row and pd.notna(row['category']) else ''
    text = str(row[doc_text_col]) if pd.notna(row[doc_text_col]) else ''
    return f"Title: {title} | Category: {cat} | Content: {text}".strip()

doc_texts = [build_doc_string(row) for _, row in documents_df.iterrows()]

# -------------------------------------------------------------------------
# 2. LEXICAL INDEXING (BM25)
# -------------------------------------------------------------------------
print("📚 Indexing Corpus with BM25...")
tokenized_corpus = [doc.lower().split() for doc in doc_texts]
bm25 = BM25Okapi(tokenized_corpus)

# -------------------------------------------------------------------------
# 3. DENSE INDEXING (BGE-LARGE & BGE-M3)
# -------------------------------------------------------------------------
print("🤖 Encoding Corpus with 'BAAI/bge-large-en-v1.5'...")
bi_large = SentenceTransformer('BAAI/bge-large-en-v1.5', device=device)
with torch.autocast(device_type=device, dtype=torch.float16):
    doc_emb_large = bi_large.encode(doc_texts, convert_to_tensor=True, show_progress_bar=True, batch_size=32)

print("🤖 Encoding Corpus with 'BAAI/bge-m3'...")
bi_m3 = SentenceTransformer('BAAI/bge-m3', device=device)
with torch.autocast(device_type=device, dtype=torch.float16):
    doc_emb_m3 = bi_m3.encode(doc_texts, convert_to_tensor=True, show_progress_bar=True, batch_size=16)

# -------------------------------------------------------------------------
# 4. LOAD CROSS-ENCODERS
# -------------------------------------------------------------------------
print("🎯 Loading Dual Rerankers...")
reranker_large = CrossEncoder('BAAI/bge-reranker-large', device=device)
reranker_base = CrossEncoder('BAAI/bge-reranker-base', device=device)

# -------------------------------------------------------------------------
# 5. HYDE / SYNTHETIC QUERY EXPANSION HELPER
# -------------------------------------------------------------------------
def generate_hyde_expansion(query):
    """
    Transforms user agricultural queries into a hypothetical technical 
    extension recommendation doc to align vector space semantics.
    """
    return f"Agricultural advice and management practices regarding {query}. Recommended control methods, optimal application rates, disease prevention, and agronomic management instructions."

# -------------------------------------------------------------------------
# 6. FUSION & ENSEMBLE EXECUTION
# -------------------------------------------------------------------------
def weighted_rrf(rank_lists, weights=[0.8, 1.2, 1.2], k=20):
    rrf_map = {}
    for rank_list, w in zip(rank_lists, weights):
        for rank, doc_idx in enumerate(rank_list):
            rrf_map[doc_idx] = rrf_map.get(doc_idx, 0.0) + (w / (k + rank + 1))
    return [doc_idx for doc_idx, _ in sorted(rrf_map.items(), key=lambda x: x[1], reverse=True)]

def normalize_scores(scores):
    s = np.array(scores)
    s_min, s_max = s.min(), s.max()
    return np.ones_like(s) if (s_max - s_min) == 0 else (s - s_min) / (s_max - s_min)

test_queries_clean = test_queries_df[test_query_col].fillna('').astype(str).str.strip().tolist()
instruction_large = "Represent this sentence for searching relevant passages: "

submission_rows = []
print("⚡ Executing HyDE Expansion + Retrieval + Dual Reranking...")

for query_id, raw_q in zip(test_queries_df[query_id_col_test], test_queries_clean):
    # Construct HyDE Expanded Text
    hyde_text = generate_hyde_expansion(raw_q)
    
    # BM25 Search (Using both original + expansion terms)
    combined_lexical_q = f"{raw_q} {hyde_text}".lower().split()
    bm25_scores = bm25.get_scores(combined_lexical_q)
    top_bm25 = np.argsort(bm25_scores)[::-1][:120]
    
    # BGE-Large Search (Averaging embeddings of original and HyDE queries)
    q_emb_orig_large = bi_large.encode(instruction_large + raw_q, convert_to_tensor=True)
    q_emb_hyde_large = bi_large.encode(instruction_large + hyde_text, convert_to_tensor=True)
    q_emb_large = (0.6 * q_emb_orig_large) + (0.4 * q_emb_hyde_large)
    
    scores_large = util.cos_sim(q_emb_large, doc_emb_large)[0].cpu().numpy()
    top_large = np.argsort(scores_large)[::-1][:120]
    
    # BGE-M3 Search (Averaging embeddings of original and HyDE queries)
    q_emb_orig_m3 = bi_m3.encode(raw_q, convert_to_tensor=True)
    q_emb_hyde_m3 = bi_m3.encode(hyde_text, convert_to_tensor=True)
    q_emb_m3 = (0.6 * q_emb_orig_m3) + (0.4 * q_emb_hyde_m3)
    
    scores_m3 = util.cos_sim(q_emb_m3, doc_emb_m3)[0].cpu().numpy()
    top_m3 = np.argsort(scores_m3)[::-1][:120]
    
    # Weighted RRF Fusion
    fused_candidates = weighted_rrf(
        [top_bm25, top_large, top_m3],
        weights=[0.8, 1.2, 1.2],
        k=20
    )[:120]
    
    # Reranking using Original User Query for precise relevance target
    pairs = [[raw_q, doc_texts[idx]] for idx in fused_candidates]
    
    with torch.autocast(device_type=device, dtype=torch.float16):
        scores_rk_large = reranker_large.predict(pairs, batch_size=64, show_progress_bar=False)
        scores_rk_base = reranker_base.predict(pairs, batch_size=64, show_progress_bar=False)
    
    norm_rk_large = normalize_scores(scores_rk_large)
    norm_rk_base = normalize_scores(scores_rk_base)
    ensemble_scores = (0.55 * norm_rk_large) + (0.45 * norm_rk_base)
    
    top_10_offsets = np.argsort(ensemble_scores)[::-1][:10]
    final_indices = [fused_candidates[i] for i in top_10_offsets]
    top_doc_ids = [documents_df.iloc[idx][doc_id_col] for idx in final_indices]
    
    for doc_id in top_doc_ids:
        submission_rows.append({
            'QueryId': query_id,
            'DocumentId': doc_id
        })

submission_df = pd.DataFrame(submission_rows)
submission_df.to_csv('submission.csv', index=False)

print("==========================================")
print("✅ Phase 2: HyDE Expansion Pipeline Complete!")
print(f"Shape: {submission_df.shape}")

🚀 Starting Phase 2 Pipeline: HyDE + Multi-Query Expansion + Dual Reranker...
⚡ Device initialized: cuda
📚 Indexing Corpus with BM25...
🤖 Encoding Corpus with 'BAAI/bge-large-en-v1.5'...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

⚡ Executing HyDE Expansion + Retrieval + Dual Reranking...
✅ Phase 2: HyDE Expansion Pipeline Complete!
Shape: (2000, 2)
